# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sanikawaghmode-mumbai/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [1]:
print("Section 1 complete.")
print("Reviewed two paper findings for:")
print("1. Label/outcome definition")
print("2. Validation design")

Section 1 complete.
Reviewed two paper findings for:
1. Label/outcome definition
2. Validation design


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
import pandas as pd
import numpy as np

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
from google.colab import files

uploaded = files.upload()

filename = next(iter(uploaded))

df = pd.read_csv(filename)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

In [ ]:
# Find possible client/group columns

possible_group_columns = [
    col for col in df.columns
    if any(word in col.lower()
           for word in ["client", "customer", "account", "site", "domain"])
]

print("Possible client/group columns:")
print(possible_group_columns)

In [ ]:
# Set the client/group column.
# If your dataset uses another name, replace "client_id".

GROUP_COLUMN = "client_id"

print("Group column:", GROUP_COLUMN)
print("Unique groups:", df[GROUP_COLUMN].nunique())

In [ ]:
# Show numerical columns

numeric_columns = df.select_dtypes(include="number").columns.tolist()

print("Numerical columns:")
for col in numeric_columns:
    print("-", col)

In [ ]:
# Candidate numerical signals
# ID-like columns are excluded.

feature_columns = [
    col for col in numeric_columns
    if not any(word in col.lower()
               for word in ["id", "index"])
]

print("Candidate signal columns:")
for col in feature_columns:
    print("-", col)

In [ ]:
# IMPORTANT:
# Replace this with the actual ranking-related outcome used in Week 4/5.

TARGET_COLUMN = "YOUR_TARGET_COLUMN"

print("Target column:", TARGET_COLUMN)

In [ ]:
# Original Week-5 style analysis

selected = df[
    feature_columns + [TARGET_COLUMN]
].dropna().copy()

original_correlation = selected.corr(
    method="spearman",
    numeric_only=True
)

original_target_correlations = (
    original_correlation[TARGET_COLUMN]
    .drop(TARGET_COLUMN)
    .sort_values(key=abs, ascending=False)
)

print("Original Spearman correlations:")
display(original_target_correlations)

In [ ]:
# Client-grouped split

from sklearn.model_selection import GroupShuffleSplit

analysis_df = df[
    feature_columns + [TARGET_COLUMN, GROUP_COLUMN]
].dropna().copy()

X = analysis_df[feature_columns]
y = analysis_df[TARGET_COLUMN]
groups = analysis_df[GROUP_COLUMN]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups)
)

train_df = analysis_df.iloc[train_idx].copy()
test_df = analysis_df.iloc[test_idx].copy()

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Training groups:", train_df[GROUP_COLUMN].nunique())
print("Test groups:", test_df[GROUP_COLUMN].nunique())

In [ ]:
# Spearman correlations on grouped training and test data

train_correlations = (
    train_df[
        feature_columns + [TARGET_COLUMN]
    ]
    .corr(method="spearman", numeric_only=True)[TARGET_COLUMN]
    .drop(TARGET_COLUMN)
)

test_correlations = (
    test_df[
        feature_columns + [TARGET_COLUMN]
    ]
    .corr(method="spearman", numeric_only=True)[TARGET_COLUMN]
    .drop(TARGET_COLUMN)
)

comparison = pd.DataFrame({
    "original_all_data": original_target_correlations,
    "grouped_train": train_correlations,
    "grouped_test": test_correlations
})

comparison["train_test_difference"] = (
    comparison["grouped_train"] -
    comparison["grouped_test"]
).abs()

comparison = comparison.sort_values(
    "train_test_difference",
    ascending=False
)

display(comparison)

### Honest split

My Week-5 method was Spearman correlation and signal analysis for the Ranking Signal Analysis lane.

The original analysis measured associations between numerical content signals and the selected ranking-related outcome.

For this validation audit, I use a client-grouped split. This keeps observations from the same client on only one side of the evaluation.

I compare the original association measurements with the measurements from the grouped train and test groups.

The comparison is treated as measured and directional evidence. It does not establish causation or guarantee performance on future clients.



## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit

I audited the final signal set for three types of leakage:

1. **Target leakage:** a signal directly contains the outcome being analysed.
2. **Future leakage:** a signal uses information that would only become available after the prediction or measurement point.
3. **Group leakage:** observations from the same client appear on both sides of the grouped evaluation.

The purpose of this audit is to check that the measured relationships are based on information that could reasonably have been available at the relevant point in time.


In [ ]:
print("Target column:")
print(TARGET_COLUMN)

print("\nSignal columns:")
for feature in feature_columns:
    print("-", feature)

if TARGET_COLUMN in feature_columns:
    print("\nWARNING: target is directly included in the signals.")
else:
    print("\nPASS: target is not directly included in the signals.")

In [ ]:
# Look for feature names that require manual future-leakage review

leakage_words = [
    "future",
    "next",
    "after",
    "post",
    "outcome",
    "target",
    "label"
]

suspicious_features = [
    feature
    for feature in feature_columns
    if any(word in feature.lower()
           for word in leakage_words)
]

print("Features requiring manual review:")
print(suspicious_features)

In [ ]:
# Missing-value audit

print("Missing values in analysis data:")

display(
    analysis_df[
        feature_columns + [TARGET_COLUMN]
    ].isna().sum()
)

In [ ]:
# Final group leakage check

print("Training groups:", len(train_groups))
print("Test groups:", len(test_groups))
print("Overlapping groups:", len(overlap))

if len(overlap) == 0:
    print("PASS: no group overlap.")
else:
    print("WARNING: group leakage detected.")

### Leakage conclusion

The final signal set was checked for direct target leakage, possible future-information leakage, and client/group overlap.

The target was not directly included as a signal. Feature names were also checked for possible indicators of future or outcome information.

The grouped split produced separate training and test client groups with no intended overlap.

No direct target leakage was identified by these checks. Time-dependent signals still require interpretation based on when their underlying information became available.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Errors and interpretation

Because the Week-5 method is correlation and signal analysis rather than a predictive model, I do not describe individual rows as prediction failures.

Instead, I inspect signals whose measured association changes the most between the grouped training and test data.

Large changes indicate that the measured relationship may be less stable across client groups.

These observations are useful for identifying signals that need further validation. They do not establish why a relationship changes or prove that a signal causes a change in search performance.


In [ ]:
# T# Signals with the largest change between grouped training and test

failure_examples = comparison.sort_values(
    "train_test_difference",
    ascending=False
)

print("Signals with the largest change in measured association:")

display(
    failure_examples.head(10)
)his cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
# Basic descriptive check for the target

print("Target summary:")
display(
    analysis_df[TARGET_COLUMN].describe()
)

### Original claim

The selected signals determine which content will rank better.

### Rewritten claim

The analysis **observed and measured** associations between selected content signals and the ranking-related outcome. These relationships provide **directional, decision-support evidence** that can help identify signals for further investigation.

The analysis does not establish that these signals cause ranking changes, and the measured relationships may vary across clients or future periods.

### Why I changed the claim

The original statement was stronger than the evidence because the word "determine" implies causation. The analysis measures associations, so the revised claim stays within what the available data and validation design can support.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.